# Revise background error for three example datasets

This notebook runs the automated revised-error workflow on three example x1d trace-stack files in `test-data` and plots the science spectrum with the original and revised error arrays centered on Lya.

In [ ]:
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

import extraction_utils as extract_utils

In [ ]:
test_data_dir = extract_utils.find_test_data_dir()

# Update this list if you have different example files
example_files = [
    test_data_dir / "of9b05010_x1d_traces.fits",
    test_data_dir / "of9b05020_x1d_traces.fits",
    test_data_dir / "of9b05030_x1d_traces.fits",
]

for path in example_files:
    if not path.exists():
        raise FileNotFoundError(f"Missing example file: {path}")

print("Example files:")
for path in example_files:
    print(f"  - {path.name}")

In [ ]:
def plot_errors_centered_on_lya(x1d_path: Path, lya: float = 1215.67, half_window: int = 60):
    with fits.open(x1d_path) as hdul:
        data = hdul[1].data
        wavelength = extract_utils._get_column(data, "wavelength")
        flux = extract_utils._get_column(data, "flux")
        error = extract_utils._get_column(data, "error")
        error_rev = extract_utils._get_column(data, "error_revised")

        if wavelength is None or flux is None or error is None or error_rev is None:
            raise KeyError("Missing one of: wavelength, flux, error, error_revised")

        if np.ndim(wavelength) > 1:
            wavelength = wavelength[0]
        if np.ndim(flux) > 1:
            flux = flux[0]
        if np.ndim(error) > 1:
            error = error[0]
        if np.ndim(error_rev) > 1:
            error_rev = error_rev[0]

    lya_idx = int(np.nanargmin(np.abs(wavelength - lya)))
    slc = slice(max(0, lya_idx - half_window), min(len(wavelength), lya_idx + half_window))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(wavelength[slc], flux[slc], color="tab:blue", lw=1.0, label="Flux")
    ax.plot(wavelength[slc], error[slc], color="tab:orange", lw=1.0, label="Error (orig)")
    ax.plot(wavelength[slc], error_rev[slc], color="tab:green", lw=1.0, label="Error (revised)")
    ax.axvline(lya, color="k", lw=1.0, alpha=0.6)
    ax.set_xlabel("Wavelength")
    ax.set_ylabel("Flux / Error")
    ax.set_title(x1d_path.name)
    ax.legend(loc="upper right")
    plt.tight_layout()

    return fig

In [ ]:
results = []
for x1d_path in example_files:
    output_path = x1d_path.with_name(x1d_path.stem + "_revised_error" + x1d_path.suffix)
    result = extract_utils.revise_background_error_in_x1d(
        x1dfile=x1d_path,
        output_x1d=output_path,
    )
    results.append(result)
    print(f"Saved: {output_path.name}")

In [ ]:
for result in results:
    plot_errors_centered_on_lya(result["output_x1d"])